# ⚡ Strompreis-Analyse: Example Notebook

This minimal notebook shows how to use the core modules in `src/` to analyze your own electricity consumption data directly in Python.

In [ ]:
# Silence Streamlit runtime warnings outside the web app
import streamlit.logger
streamlit.logger.set_log_level("ERROR")

import pandas as pd
from src.file_parser import ConsumptionDataParser
from src.data_loader import get_spot_data, merge_consumption_with_prices
from src.tariffs import TariffManager
from src.analysis import classify_usage, compute_cost_comparison_data

## 1. Load and Parse Smart Meter Data

`ConsumptionDataParser` automatically detects and standardizes various European smart meter CSV formats into UTC timestamps and hourly/15-minute consumption.

In [ ]:
parser = ConsumptionDataParser()
with open("resources/EXAMPLE-DATA-15M.csv", "rb") as f:
    df_consumption = parser.parse_file(f)

print(f"Loaded {len(df_consumption):,} records from {df_consumption['timestamp'].min().date()} to {df_consumption['timestamp'].max().date()}.")
df_consumption.head()

## 2. Fetch and Merge EPEX Spot Prices

Retrieve historical spot market data (cached locally in `cache/`) and align it with your consumption data.

In [ ]:
# Use a 30-day window for demonstration
start_date = df_consumption["timestamp"].min().date()
end_date = (df_consumption["timestamp"].min() + pd.Timedelta(days=30)).date()

mask = (df_consumption["timestamp"].dt.date >= start_date) & (df_consumption["timestamp"].dt.date <= end_date)
df_sample = df_consumption.loc[mask]

# Fetch spot prices (Austria: 'at', Germany: 'de')
df_spot = get_spot_data(country="at", start=start_date, end=end_date)
df_merged = merge_consumption_with_prices(df_sample, df_spot)
df_merged.head()

## 3. Compare Flexible vs. Static Tariffs

Calculate exact costs including provider markups, VAT, and prorated monthly fees.

In [ ]:
manager = TariffManager("resources/tariffs_flexible.json", "resources/tariffs_static.json")
flex_tariffs = manager.get_flex_tariffs_with_custom()
static_tariffs = manager.get_static_tariffs_with_custom()

# Pick tariffs to compare (or configure your own Custom tariff)
flex_tariff = flex_tariffs["smartCONTROL (smartENERGY)"]
static_tariff = static_tariffs["V-Strom CLASSIC (VERBUND)"]

# Compute interval-by-interval costs
df_cost = manager.run_cost_analysis(df_merged.copy(), flex_tariff, static_tariff)

total_kwh = df_cost["consumption_kwh"].sum()
cost_flex = df_cost["total_cost_flexible"].sum()
cost_static = df_cost["total_cost_static"].sum()
savings = cost_static - cost_flex

print(f"Total Consumption: {total_kwh:.1f} kWh")
print(f"Static ({static_tariff.name}):   €{cost_static:.2f} (Ø €{cost_static / total_kwh:.3f}/kWh)")
print(f"Flexible ({flex_tariff.name}): €{cost_flex:.2f} (Ø €{cost_flex / total_kwh:.3f}/kWh)")
print(f"Potential Savings:                   €{savings:.2f} ({savings / cost_static * 100:+.1f}%)")

## 4. Load Classification & Aggregated Summaries

Decompose consumption into base, regular, and peak components and aggregate costs across periods.

In [ ]:
# Decompose usage into base, regular, and peak load
df_classified, base_thresh, peak_thresh = classify_usage(df_cost.copy(), "Europe/Vienna")
print(f"Base Load Total:    {df_classified['base_load_kwh'].sum():.1f} kWh")
print(f"Regular Load Total: {df_classified['regular_load_kwh'].sum():.1f} kWh")
print(f"Peak Load Total:    {df_classified['peak_load_kwh'].sum():.1f} kWh")

# Aggregate by week
weekly_summary = compute_cost_comparison_data(df_cost, resolution="Weekly")
weekly_summary